In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
from scipy.interpolate import interp1d
from matplotlib.ticker import MultipleLocator

In [ ]:
arr_load = np.load(r"d:\Chip2025_Testing\Python_Notebook\Tests\Data\FAST_IMC_SWEEP\ALLBank_1x_TD_Sweep_1_Ref256_temp.npy")
print(arr_load.shape)

for i in range(arr_load.shape[0]): #inputs
    for j in range(arr_load.shape[1]): #Bank
        for k in range(arr_load.shape[2]): #BL
            if(np.all(arr_load[i,j,k,:]==0)): #All DLs
                print("Problem",i,j,k)

arr = np.reshape(arr_load,(arr_load.shape[0],arr_load.shape[1]*arr_load.shape[2]*arr_load.shape[3]))
arr.shape



In [ ]:
print(arr_load[0,1,:,:])

In [ ]:
#ideal_mac = np.arange(0,128.5,0.5)
ideal_mac = np.arange(0,257,1)
x_axis = list(range(0,257,1))
print(arr.shape)
corrected = []
slope = []
bias = []
error = []
cmap = cm.get_cmap('Blues')
cmap2 = cm.get_cmap('Greens')
fig, ax = plt.subplots(figsize=(8, 4))
plt.title("Post Silicon Measured MAC vs Ideal MAC")
plt.xlabel("Input Code")
plt.ylabel("Output Code")
for i in range(256*0,256*4,1): #arr.shape[1]
    d1 = arr[:,i]
    m, c = np.polyfit(d1,ideal_mac , 1)
    slope.append(m)
    bias.append(c)
    d2 = np.round(d1*m+c)
    #plt.plot(ideal_mac, 2*d1, color = cmap(0.3))
    plt.plot(x_axis, d2, color = cmap2(0.7))
    corrected.append(d2)
    error.append(d2 - ideal_mac)
corrected = np.array(corrected)
plt.plot(x_axis, d2, color = cmap2(0.7), label = "Measured Mac across Delay Lines")
plt.plot(x_axis, ideal_mac,color = "Red", label = "Ideal MAC Value")
plt.legend() 
plt.grid(True)
plt.show()

In [ ]:
slope_np = np.array(slope,dtype=np.float32)
bias_np = np.array(bias,dtype=np.float32)
slope_np = np.reshape(slope_np,(4,4,64)) #BANK,BL,DL
bias_np = np.reshape(bias_np,(4,4,64))
combined = np.stack((slope_np, bias_np), axis=0) #slope and bias combined
print(slope_np)
#np.save(r'd:\Chip2025\Chip2025_Testing\Python_Notebook\Utils\DL_Correction_Data\TD_DL_CORRECTION.npy', combined)

In [ ]:
print(corrected.shape)
corrected[:64,256]

In [ ]:
print(corrected.shape)
for i in range(1024):
    for j in range(30,257):
        if(corrected[i,j] < 0) :
            print(i,j)

In [ ]:
print(list(arr_load[87,0,0,:]))
print(list(arr_load[87,0,1,:]))
print(list(arr_load[87,0,2,:]))
print(list(arr_load[87,0,3,:]))

In [ ]:
print(slope_np)

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams['font.family'] = 'Times New Roman'   # or 'Times New Roman', 'DejaVu Sans', etc.
mpl.rcParams['font.size'] = 12

# Axis-specific defaults
mpl.rcParams['axes.titlesize'] = 12
mpl.rcParams['axes.labelsize'] = 12
mpl.rcParams['xtick.labelsize'] = 12
mpl.rcParams['ytick.labelsize'] = 12
mpl.rcParams['legend.fontsize'] = 12

In [ ]:
mean_corrected = np.mean(corrected,axis=0)
std_corrected = np.std(corrected,axis=0)
#std_corrected = std_corrected[:len(std_corrected)-5]
std_corrected[len(std_corrected)-6:len(std_corrected)] = std_corrected[len(std_corrected)-6]
fig, ax1 = plt.subplots(figsize=(4, 2))

# First plot (left y-axis)
ax1.plot(x_axis, mean_corrected, color="r",linewidth=2.5)
ax1.set_xlabel("Input Code",fontsize = 12,fontname="Times New Roman")
ax1.set_ylabel("Mean Output across Macro", color="r",fontsize = 12,fontname="Times New Roman")
ax1.tick_params(axis="y", labelcolor="r")
ax1.set_ylim(0, 140.8*2)
ax1.yaxis.set_major_locator(MultipleLocator(32*2))

# Second plot (right y-axis)
ax2 = ax1.twinx()
ax2.plot( std_corrected, color="b",alpha=0.5,linewidth=2.5)
ax2.set_ylabel("Std (LSB)", color="b",fontsize = 12,fontname="Times New Roman")
ax2.tick_params(axis="y", labelcolor="b")
ax2.set_ylim(0, 2.2)
ax2.yaxis.set_major_locator(MultipleLocator(0.5))



plt.xticks(range(0, 257,32)) 
# Optional: title
plt.title("TD MAC Linearity and Mismatch (Measured)",fontsize = 12,fontname="Times New Roman")
ax1.grid(True, which='major', linestyle='--', linewidth=1, alpha=0.7)
ax1.grid(True, which='minor', linestyle=':', linewidth=1, alpha=0.5)
#plt.savefig("D:\ISSCC_26\Graphs\TD_2x_Acc_109.jpg", dpi=800, bbox_inches="tight")
plt.show()

In [ ]:
print(np.mean(std_corrected))
corrected.shape

In [ ]:
corrected_mac = np.reshape(corrected,(4,4,64,257))
HW_MODEL_TD_1x = np.mean(corrected_mac,axis=1)
print(HW_MODEL_TD_1x.shape)
#np.save("d:\Chip2025\Chip2025_Testing\Python_Notebook\Tests\Data\Hardware_Model\HW_MODEL_TD_1x_FastIMC.npy",HW_MODEL_TD_1x)

In [ ]:
print(corrected_mac[0,:,:,7])

In [ ]:
print(HW_MODEL_TD_1x[:,:,0])

In [ ]:
corrected_mac = np.transpose(corrected)
corrected_mac = np.reshape(corrected_mac,(257,4,1,64))
corrected_mac = corrected_mac[:,:,0,:]
corrected_mac.shape


In [ ]:
HW_MODEL_TD_1x = np.zeros(((4,64,257)),dtype=np.float32)
x_new = np.arange(0, 257, 1)   # length 257
x_orig = np.arange(0, 257, 1)  # length 65
# Interpolation
for bank in range(4):
    for dl in range(64):
        a = corrected_mac[:,bank,dl]
        HW_MODEL_TD_1x[bank,dl] = a #np.interp(x_new, x_orig, a)
        plt.plot(HW_MODEL_TD_1x[bank,dl])
#np.save("d:\Chip2025\Chip2025_Testing\Python_Notebook\Tests\Data\Hardware_Model\HW_MODEL_TD_1x.npy",HW_MODEL_TD_1x)

In [ ]:
print(HW_MODEL_TD_1x[0,9])

In [ ]:
sigma = []
for i in range(0,arr.shape[1],1):
    plt.plot(ideal_mac, error[i], label='Data', color = cmap2(0.7))
    sigma.append(np.std(error[i]))
print(np.mean(sigma))

In [ ]:
def compute_dnl_inl(actual_2d, ideal_1d):
    ideal_diff = np.diff(ideal_1d)
    ideal_lsb = np.mean(ideal_diff)

    # Compute DNL: difference of step sizes from ideal LSB
    dnl = (np.diff(actual_2d, axis=1) / ideal_lsb) -1

    # Compute INL: deviation from ideal (cumulative DNL)
    inl = np.cumsum(dnl, axis=1)

    # Pad DNL and INL to match original array length
    dnl = np.pad(dnl, ((0, 0), (0, 1)), mode='edge')
    inl = np.pad(inl, ((0, 0), (0, 1)), mode='edge')

    return dnl, inl

In [ ]:
dnl,inl = compute_dnl_inl(corrected,ideal_mac)

In [ ]:
mean_inl = np.mean(inl, axis=0)
plt.figure(figsize=(8, 4))
plt.title("MAC INL across all Delay Lines")
plt.xlabel("Ideal Output Code")
plt.ylabel("INL(LSB)")
cmap = cm.get_cmap('Blues')
# Standard deviation (sigma) per code index
sigma_inl = np.std(inl, axis=0)
plus_3sigma = mean_inl + 3 * sigma_inl
minus_3sigma = mean_inl - 3 * sigma_inl
for i,j in zip(inl,dnl):
    plt.plot(ideal_mac, i, color=cmap(0.3))
plt.plot(ideal_mac, mean_inl, color = "red",label = "Mean")
plt.plot(ideal_mac, plus_3sigma, color = "blue",label = "3 sigma")
plt.plot(ideal_mac, minus_3sigma,color = "blue")
plt.legend() 
plt.grid(True)
plt.show()


In [ ]:
mean_dnl = np.mean(dnl, axis=0)
plt.figure(figsize=(8, 4))
plt.title("MAC DNL across all Delay Lines")
plt.xlabel("Ideal Output Code")
plt.ylabel("DNL(LSB)")
# Standard deviation (sigma) per code index
sigma_dnl = np.std(dnl, axis=0)
plus_3sigma = mean_dnl + 3 * sigma_dnl
minus_3sigma = mean_dnl - 3 * sigma_dnl
for i,j in zip(inl,dnl):
    plt.plot(ideal_mac, j,color=cmap(0.3))
plt.plot(ideal_mac, mean_dnl, color = "red",label = "Mean")
plt.plot(ideal_mac, plus_3sigma,color = "blue",label = "3 sigma" )
plt.plot(ideal_mac, minus_3sigma,color = "blue")
plt.legend() 
plt.grid(True)
plt.show()